In [1]:
# ===== NOTEBOOK 2 - STEP 1: load the merged dataset + make the 5 folds BY PATIENT =====
import os, glob, time, gc, pickle
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold

SEED, N_FOLDS = 42, 5
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

# ---- 1) Find the output of notebook 1 ----
META = glob.glob("/kaggle/input/**/metadata_final.csv", recursive=True)
print("Found:", META)
DATA_DIR = os.path.dirname(META[0])
data = pd.read_csv(META[0])
print(f"{len(data)} images | {data['patient_id'].nunique()} patients")


# ---- 2) Load all images (ONE .npz file: row i = row i of metadata_final.csv) ----
X_full = np.load(glob.glob(f"{DATA_DIR}/images_*.npz")[0])["X"]
assert len(X_full) == len(data), "images and metadata do not match!"
print("X:", X_full.shape, X_full.dtype, f"({X_full.nbytes / 1e9:.2f} GB)")

# ---- 3) 5 folds BY PATIENT, stratified by class AND dataset ----
# (every fold gets the same mix of datasets and classes -> fair comparison)
def make_folds(df, label_col):
    strata = df["dataset"] + "_" + df[label_col].astype(str)
    best_score, best_seed, best_folds = 1e9, None, None
    for seed in range(100):                                   # try 100 seeds, keep the most balanced
        sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        folds = np.zeros(len(df), dtype=int)
        for f, (_, te) in enumerate(sgkf.split(df, strata, groups=df["patient_id"])):
            folds[te] = f
        score = pd.crosstab(folds, strata, normalize="index").std().sum()
        if score < best_score:
            best_score, best_seed, best_folds = score, seed, folds
    return best_seed, best_folds

data["fold_multi"], data["fold_bin"] = -1, -1
seed_multi, data["fold_multi"] = make_folds(data, "label_multi")

is_bin = data["label_bin"] >= 0                               # binary task: Benign + Malignant only
seed_bin, folds_bin = make_folds(data[is_bin].reset_index(drop=True), "label_bin")
data.loc[is_bin, "fold_bin"] = folds_bin
print(f"Best seeds -> multiclass: {seed_multi} | binary: {seed_bin}")

# ---- 4) Checks: no patient in 2 folds + mix of datasets/classes per fold ----
for task in ["multi", "bin"]:
    d = data[data[f"fold_{task}"] >= 0]
    leak = (d.groupby("patient_id")[f"fold_{task}"].nunique() > 1).sum()
    print(f"\n[{task}] patients in more than one fold: {leak}")
    t = pd.crosstab(d[f"fold_{task}"], [d["dataset"], d["label_name"]])
    t["TOTAL"] = t.sum(axis=1)
    display(t)

data.to_csv("/kaggle/working/data_folds.csv", index=False)


TensorFlow 2.20.0 | GPUs: 2
Found: ['/kaggle/input/datasets/mohamedbaounna/mammo-merged-packed/packed/metadata_final.csv']
2204 images | 1091 patients
X: (2204, 448, 448) uint8 (0.44 GB)
Best seeds -> multiclass: 35 | binary: 46

[multi] patients in more than one fold: 0


dataset    CBIS-DDSM           INbreast                    MIAS            \
label_name    Benign Malignant   Benign Malignant Normal Benign Malignant   
fold_multi                                                                  
0                142       146        8        10     65     16         9   
1                149       154        0        28     54     12        11   
2                148       148        7        24     58      7        14   
3                140       155        2        20     62     14         7   
4                147       143        6        18     48     14        11   

dataset           TOTAL  
label_name Normal        
fold_multi               
0              41   437  
1              41   449  
2              43   449  
3              41   441  
4              41   428


[bin] patients in more than one fold: 0


dataset    CBIS-DDSM           INbreast             MIAS           TOTAL
label_name    Benign Malignant   Benign Malignant Benign Malignant      
fold_bin                                                                
0                149       139        2        20      9        11   330
1                147       148        6        20     14         8   343
2                143       153        6        18     10        13   343
3                143       158        5        20     14        10   350
4                144       148        4        22     16        10   344

In [2]:
# ===== NOTEBOOK 2 - STEP 2: training functions + smoke test + timing test =====
from tensorflow.keras import layers, Model, regularizers
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight

AUTOTUNE    = tf.data.AUTOTUNE
IMG_SIZE    = 448
EPOCHS_HEAD = 10          # phase 1: CNN frozen, only the new head learns
EPOCHS_FT   = 25          # phase 2: whole CNN fine-tuned (small learning rate)
LR_HEAD, LR_FT = 1e-3, 5e-5
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- Speed: mixed precision + the 2 GPUs together ----
tf.keras.mixed_precision.set_global_policy("mixed_float16")
USE_2_GPUS = True
strategy   = tf.distribute.MirroredStrategy() if USE_2_GPUS else tf.distribute.get_strategy()
BATCH_SIZE = 16 * strategy.num_replicas_in_sync
print("GPUs used:", strategy.num_replicas_in_sync, "| batch size:", BATCH_SIZE)

X = X_full if IMG_SIZE == 448 else np.stack(
    [cv2.resize(x, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA) for x in X_full])

# ---- 1) The 5 CNNs + the input format each one expects ----
BACKBONES = {
    "VGG16":          (tf.keras.applications.VGG16,          "caffe"),
    "ResNet50":       (tf.keras.applications.ResNet50,       "caffe"),
    "DenseNet121":    (tf.keras.applications.DenseNet121,    "torch"),
    "EfficientNetB0": (tf.keras.applications.EfficientNetB0, "raw"),
    "MobileNetV2":    (tf.keras.applications.MobileNetV2,    "tf"),
}

def normalize(x, mode):
    if mode == "caffe":   # VGG16, ResNet50
        return x - tf.constant([103.939, 116.779, 123.68])
    if mode == "torch":   # DenseNet121
        return (x / 255.0 - tf.constant([0.485, 0.456, 0.406])) / tf.constant([0.229, 0.224, 0.225])
    if mode == "tf":      # MobileNetV2
        return x / 127.5 - 1.0
    return x              # EfficientNetB0 (normalizes inside the model)

# ---- 2) Data pipeline (+ augmentation with BLACK borders) ----
def make_augmenter():
    return tf.keras.Sequential([
        layers.RandomRotation(5 / 360, fill_mode="constant", fill_value=0.0),
        layers.RandomZoom(0.1, fill_mode="constant", fill_value=0.0),
        layers.RandomTranslation(0.1, 0.1, fill_mode="constant", fill_value=0.0),
        layers.RandomContrast(0.1),
    ])

def make_dataset(rows, y, mode, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X[rows], y))
    if shuffle:
        ds = ds.shuffle(len(rows), seed=SEED)
    ds = ds.map(lambda a, b: (tf.repeat(tf.cast(a, tf.float32)[..., None], 3, axis=-1), b),
                num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE, drop_remainder=shuffle)       # grey -> 3 channels
    if augment:
        aug = make_augmenter()
        ds = ds.map(lambda a, b: (tf.cast(aug(a, training=True), tf.float32), b),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda a, b: (normalize(a, mode), b), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

# ---- 3) Model = pretrained CNN + new head ----
def build_model(name, n_classes):
    constructor, _ = BACKBONES[name]
    base = constructor(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)                 # BatchNorm stays in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="softmax", dtype="float32")(x)
    return Model(inputs, outputs, name=name), base

# ---- 4) Real validation AUC (sklearn) -> used by EarlyStopping ----
class ValAUC(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val, n_classes):
        super().__init__()
        self.val_ds, self.y_val, self.n_classes = val_ds, y_val, n_classes
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.val_ds, verbose=0)
        try:
            auc = (roc_auc_score(self.y_val, p[:, 1]) if self.n_classes == 2
                   else roc_auc_score(self.y_val, p, multi_class="ovr", average="macro"))
        except ValueError:
            auc = 0.5
        logs["val_auc_true"] = auc

# ---- 5) Metrics ----
def best_threshold(y, p):                     # binary: best TPR - FPR on VALIDATION
    fpr, tpr, thr = roc_curve(y, p)
    t = thr[np.argmax(tpr - fpr)]
    return float(t) if np.isfinite(t) else 0.5

def compute_metrics(y_true, y_prob, n_classes, thr=None):
    if n_classes == 2:
        y_pred = (y_prob[:, 1] >= (0.5 if thr is None else thr)).astype(int)
        auc = roc_auc_score(y_true, y_prob[:, 1])
    else:
        y_pred = y_prob.argmax(1)
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    last = n_classes - 1                       # Malignant is the last class in both tasks
    return {
        "accuracy":     accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision":    precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall":       recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1":           f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auc":          auc,
        "sensitivity":  recall_score(y_true == last, y_pred == last, zero_division=0),
    }

# ---- 6) Train + test ONE model on ONE fold ----
def run_one(model_name, task, fold, augment, verbose=0, limit=None, epochs=None):
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    e_head, e_ft = epochs or (EPOCHS_HEAD, EPOCHS_FT)

    d         = data[data[f"fold_{task}"] >= 0]
    rows      = d.index.values                              # row numbers in X
    labels    = d[f"label_{task}"].values.astype(int)
    folds     = d[f"fold_{task}"].values
    patients  = d["patient_id"].values
    strata    = (d["dataset"] + "_" + d[f"label_{task}"].astype(str)).values
    n_classes = 2 if task == "bin" else 3
    onehot    = np.eye(n_classes, dtype=np.float32)[labels]
    _, mode   = BACKBONES[model_name]

    # Test = this fold | validation = 20% of the other PATIENTS | train = the rest
    test     = np.where(folds == fold)[0]
    trainval = np.where(folds != fold)[0]
    inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr, va = next(inner.split(trainval, strata[trainval], groups=patients[trainval]))
    train, val = trainval[tr], trainval[va]
    if limit:                                                # small subset (smoke test only)
        rng = np.random.RandomState(0)
        train, val, test = (rng.permutation(train)[:limit], rng.permutation(val)[:limit // 2],
                            rng.permutation(test)[:limit // 2])

    train_ds = make_dataset(rows[train], onehot[train], mode, augment=augment, shuffle=True)
    val_ds   = make_dataset(rows[val],   onehot[val],   mode)
    test_ds  = make_dataset(rows[test],  onehot[test],  mode)

    cw = compute_class_weight("balanced", classes=np.arange(n_classes), y=labels[train])
    class_weight = dict(enumerate(cw))                       # rare classes count more

    def callbacks():
        return [ValAUC(val_ds, labels[val], n_classes),      # must be FIRST
                tf.keras.callbacks.EarlyStopping(monitor="val_auc_true", mode="max", patience=8,
                                                 restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc_true", mode="max", factor=0.5,
                                                     patience=3, min_lr=1e-7)]

    t0 = time.time()
    with strategy.scope():                                   # Phase 1: CNN frozen
        model, base = build_model(model_name, n_classes)
        model.compile(optimizer=tf.keras.optimizers.Adam(LR_HEAD),
                      loss="categorical_crossentropy", metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=e_head,
                   class_weight=class_weight, callbacks=callbacks(), verbose=verbose)
    with strategy.scope():                                   # Phase 2: fine-tune everything
        base.trainable = True
        model.compile(optimizer=tf.keras.optimizers.Adam(LR_FT),
                      loss="categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=e_ft,
                   class_weight=class_weight, callbacks=callbacks(), verbose=verbose)
    train_time = time.time() - t0

    thr = None
    if n_classes == 2:                                       # threshold chosen on validation
        thr = best_threshold(labels[val], model.predict(val_ds, verbose=0)[:, 1])

    t1 = time.time()
    y_prob = model.predict(test_ds, verbose=0)
    infer_ms = 1000 * (time.time() - t1) / len(test)
    y_true = labels[test]

    m = compute_metrics(y_true, y_prob, n_classes, thr)
    m.update({"model": model_name, "task": task, "fold": fold, "augment": augment,
              "threshold": thr, "n_train": len(train), "params_M": model.count_params() / 1e6,
              "train_time_s": train_time, "infer_ms_per_img": infer_ms,
              "epochs_run": len(h1.history["loss"]) + len(h2.history["loss"])})
    history = {k: h1.history[k] + h2.history.get(k, []) for k in h1.history}
    return m, history, rows[test], y_true, y_prob

print("Functions ready ✅\n")




INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs used: 2 | batch size: 32
Functions ready ✅



I0000 00:00:1792749420.223570      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792749420.226969      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
# ===== NOTEBOOK 2 - STEP 2b: protection against NaN (exploding gradients) =====
# 1) Gradient clipping: limits the size of each weight update -> training cannot "explode"
if "_ADAM_ORIG" not in globals():
    _ADAM_ORIG = tf.keras.optimizers.Adam
def Adam_clipped(learning_rate=1e-3, **kw):
    kw.setdefault("clipnorm", 1.0)
    return _ADAM_ORIG(learning_rate, **kw)
tf.keras.optimizers.Adam = Adam_clipped            # run_one now uses the clipped Adam

# 2) Detect NaN predictions
if "_best_threshold_orig" not in globals():
    _best_threshold_orig, _compute_metrics_orig = best_threshold, compute_metrics
def best_threshold(y, p):
    if not np.all(np.isfinite(p)):
        raise FloatingPointError("NaN in validation predictions")
    return _best_threshold_orig(y, p)
def compute_metrics(y_true, y_prob, n_classes, thr=None):
    if not np.all(np.isfinite(y_prob)):
        raise FloatingPointError("NaN in test predictions")
    return _compute_metrics_orig(y_true, y_prob, n_classes, thr)

# 3) Try float16 (fast), then float32; if both fail -> record "failed" and CONTINUE
def run_safe(model_name, task, fold, augment):
    for policy in ["mixed_float16", "float32"]:
        tf.keras.mixed_precision.set_global_policy(policy)
        try:
            out = run_one(model_name, task, fold, augment)
            out[0]["precision_mode"] = policy
            return out
        except FloatingPointError as e:
            print(f"   ⚠️ {model_name} aug={augment} fold={fold} [{policy}]: {e}", flush=True)
        finally:
            tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"   ❌ {model_name} aug={augment} fold={fold}: failed twice -> skipped", flush=True)
    m = {k: np.nan for k in ["accuracy", "balanced_acc", "precision", "recall", "f1",
                             "auc", "sensitivity", "train_time_s"]}
    m.update({"model": model_name, "task": task, "fold": fold, "augment": augment,
              "precision_mode": "failed"})
    n_classes = 2 if task == "bin" else 3
    return m, {}, np.array([], dtype=int), np.array([]), np.empty((0, n_classes))

print("NaN protection ready ✅ (gradient clipping + float32 retry + skip on failure)")


NaN protection ready ✅ (gradient clipping + float32 retry + skip on failure)


In [4]:
# ===== NOTEBOOK 2 - STEP 3: train ALL models for ONE task (saved after each training) =====
tf.get_logger().setLevel("ERROR")                      # hide the INFO / WARNING messages

TASK = "bin"       # Version 1: "bin"   |   Version 2: "multi"
IMG_SIZE = 384               # 448 -> 384: faster, still 3x more detail than 224
EPOCHS_HEAD, EPOCHS_FT = 5, 15
X = np.stack([cv2.resize(x, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA) for x in X_full])

MODELS = ["MobileNetV2", "EfficientNetB0", "ResNet50", "DenseNet121", "VGG16"]   # fastest first
TIME_BUDGET_H = 11.0         # stop safely before Kaggle's 12h limit

results_csv = f"{RESULTS_DIR}/results_{TASK}.csv"
preds_pkl   = f"{RESULTS_DIR}/predictions_{TASK}.pkl"

# Resume: if a previous version already has some results (added as input), continue from them
results, preds = [], {}
prev = glob.glob(f"/kaggle/input/**/results_{TASK}.csv", recursive=True)
if prev:
    results = pd.read_csv(prev[0]).to_dict("records")
    results = [r for r in results if r.get("precision_mode") != "failed"]   # re-train the failed ones

    with open(os.path.join(os.path.dirname(prev[0]), f"predictions_{TASK}.pkl"), "rb") as f:
        preds = pickle.load(f)
    print(f"Resuming: {len(results)} trainings already done")
done = {(r["model"], bool(r["augment"]), int(r["fold"])) for r in results}

jobs = [(mn, aug, f) for mn in MODELS for aug in [False, True] for f in range(N_FOLDS)]
start = time.time()
for k, (model_name, augment, fold) in enumerate(jobs, 1):
    if (model_name, augment, fold) in done:
        continue
    if (time.time() - start) / 3600 > TIME_BUDGET_H:
        print("⏰ Time budget reached -> stopping. Results are saved; run a new version to continue.")
        break

    m, hist, test_rows, yt, yp = run_safe(model_name, TASK, fold, augment)
    m["img_size"] = IMG_SIZE
    results.append(m)
    preds[(model_name, augment, fold)] = {"rows": test_rows, "y_true": yt, "y_prob": yp, "history": hist}

    # save after EACH training -> nothing is lost
    pd.DataFrame(results).to_csv(results_csv, index=False)
    with open(preds_pkl, "wb") as f:
        pickle.dump(preds, f)

    print(f"[{k:3d}/{len(jobs)}] {model_name:15s} aug={str(augment):5s} fold={fold} | "
          f"AUC={m['auc']:.3f} bal_acc={m['balanced_acc']:.3f} F1={m['f1']:.3f} | "
          f"{m['train_time_s']/60:.1f} min | total {(time.time() - start)/3600:.2f} h", flush=True)

print("\nDONE ✅")
summary = (pd.DataFrame(results).groupby(["model", "augment"])
           [["auc", "balanced_acc", "f1", "sensitivity", "train_time_s"]]
           .agg(["mean", "std"]).round(3))
display(summary)


Resuming: 42 trainings already done


/tmp/ipykernel_22/1118467657.py:68: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = constructor(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[  5/50] MobileNetV2     aug=False fold=4 | AUC=0.592 bal_acc=0.574 F1=0.574 | 4.0 min | total 0.07 h


/tmp/ipykernel_22/1118467657.py:68: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = constructor(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))


[ 10/50] MobileNetV2     aug=True  fold=4 | AUC=0.571 bal_acc=0.533 F1=0.524 | 6.9 min | total 0.18 h
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


E0000 00:00:1792750101.885407      22 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetB0_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
E0000 00:00:1792750191.440817      22 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetB0_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


[ 15/50] EfficientNetB0  aug=False fold=4 | AUC=0.669 bal_acc=0.638 F1=0.638 | 7.5 min | total 0.31 h


E0000 00:00:1792750564.137855      22 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetB0_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
E0000 00:00:1792750712.259428      22 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetB0_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


[ 20/50] EfficientNetB0  aug=True  fold=4 | AUC=0.603 bal_acc=0.560 F1=0.554 | 9.5 min | total 0.47 h
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[ 25/50] ResNet50        aug=False fold=4 | AUC=0.679 bal_acc=0.626 F1=0.604 | 11.1 min | total 0.66 h
[ 30/50] ResNet50        aug=True  fold=4 | AUC=0.603 bal_acc=0.567 F1=0.564 | 12.7 min | total 0.87 h
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[ 35/50] DenseNet121     aug=False fold=4 | AUC=0.566 bal_acc=0.526 F1=0.525 | 9.7 min | total 1.04 h
[ 40/50] DenseNet121     aug=True  fold=4 | AUC=0.570 bal_acc=0.571 F1=0.570 | 13.2 min | total 1.26 h

DONE ✅


auc        balanced_acc            f1         \
                         mean    std         mean    std   mean    std   
model          augment                                                   
DenseNet121    False    0.633  0.048        0.588  0.050  0.580  0.055   
               True     0.634  0.045        0.601  0.033  0.593  0.033   
EfficientNetB0 False    0.658  0.018        0.610  0.028  0.604  0.037   
               True     0.617  0.015        0.564  0.014  0.551  0.023   
MobileNetV2    False    0.599  0.033        0.572  0.026  0.569  0.026   
               True     0.618  0.030        0.589  0.035  0.580  0.033   
ResNet50       False    0.689  0.041        0.628  0.040  0.613  0.049   
               True     0.646  0.034        0.605  0.030  0.598  0.028   
VGG16          False    0.628  0.078        0.594  0.047  0.581  0.059   
               True     0.614  0.034        0.577  0.033  0.566  0.050   

                       sensitivity        train_time_s           
                              mean    std         mean      std  
model          augment                                           
DenseNet121    False         0.550  0.153      676.353  135.916  
               True          0.512  0.079      881.775   61.066  
EfficientNetB0 False         0.615  0.144      447.625   10.176  
               True          0.553  0.187      517.032   69.630  
MobileNetV2    False         0.532  0.062      286.051   50.555  
               True          0.569  0.128      440.036   60.250  
ResNet50       False         0.486  0.117      642.655   37.390  
               True          0.569  0.110      758.467    6.760  
VGG16          False         0.484  0.148      556.676   77.096  
               True          0.629  0.135      629.997   74.250